# 02 — Pipeline & Inferenz

Phase 3 (Pipeline bauen + erste Inferenz auf 12 Hand-Gold-Anzeigen), Phase 4 (Iterationen A + B — pro Iteration eigener Predictions-Dateiname und Run-Header-Update), Phase 6 (voller Korpus auf 7B + 3B-Kontrast auf euler).

Cheatsheets: `CHEATSHEETS/transformers-konzepte.md` (Modell, Chat-Template, JSON-Parsing), `CHEATSHEETS/gpu-zugang.md` (Spawn, GPU-Wahl, Memory).

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _YYYY-MM-DD_ |
| Modell | _ (z. B. `Qwen/Qwen2.5-7B-Instruct`) |
| Server | _ (gauss / euler) |
| GPU-Index | _ |
| Schema-Datei | `SCHEMA.md` |
| Aktueller Run-Tag | _ (`baseline` / `iter_A` / `iter_B` / `full_7b` / `full_3b`) |
| Predictions-Datei | _ (`predictions.jsonl` / `predictions_iter_A.jsonl` / …) |
| Truncation | _ Zeichen (initial ~2000) |

Bei jeder neuen Iteration: Run-Tag + Predictions-Datei + Datum aktualisieren.

## Phase 3 — Pipeline bauen + Baseline-Inferenz

In [1]:
import os

# 1. Parameter definieren
RUN_INFO = {
    "modell":             "Qwen/Qwen2.5-3B-Instruct",
    "dtype":              "torch.float16",
    "server":             "euler",
    "gpus":               "1,2,3",  # <-- HIER: Flexible Auswahl (z. B. "0,1,2,3" oder "2,3" oder "0")
    "datum":              "2026-05-11",
    "phase":              "3.1 — Pipeline + erste Inferenz auf 12 Hand-Gold",
    "predictions_datei":  "daten/predictions_baseline.jsonl",
}

# 2. GPUs SOFORT sperren/auswählen (MUSS VOR 'import torch' PASSIEREN!)
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = RUN_INFO["gpus"]

for schluessel, wert in RUN_INFO.items():
    print(f"{schluessel:22} {wert}")

modell                 Qwen/Qwen2.5-3B-Instruct
dtype                  torch.float16
server                 euler
gpus                   1,2,3
datum                  2026-05-11
phase                  3.1 — Pipeline + erste Inferenz auf 12 Hand-Gold
predictions_datei      daten/predictions_baseline.jsonl


In [3]:
# imports + Pfade
import json
import re
import time
import subprocess
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

basispfad        = Path("/home/jovyan/work/notebooks/LLM-Workshop/llm-workshop")
korpus_pfad      = basispfad / "daten" / "eigener_korpus.jsonl"
gold_pfad        = basispfad / "AUFGABEN" / "annotation" / "meine_gold.csv"
predictions_pfad = basispfad / RUN_INFO["predictions_datei"]
predictions_pfad.parent.mkdir(parents=True, exist_ok=True)
print(f"Predictions werden geschrieben nach: {predictions_pfad}")

<jemalloc>: Unsupported system page size


Predictions werden geschrieben nach: /home/jovyan/work/notebooks/LLM-Workshop/llm-workshop/daten/predictions_baseline.jsonl


In [4]:
# 2. Modell laden 
print(f"Lade {RUN_INFO['modell']} auf {torch.cuda.device_count()} GPUs...")
    
modell = AutoModelForCausalLM.from_pretrained(
    RUN_INFO["modell"],
    torch_dtype=torch.float16,
    device_map="auto"
    
)

    
print("Abgeschlossen.")

Lade Qwen/Qwen2.5-3B-Instruct auf 3 GPUs...


/opt/conda/envs/torch/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/opt/conda/envs/torch/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Abgeschlossen.


In [5]:
## Daten laden + 12 Anzeigen aus csv joinen
korpus_eintraege = []
with open(korpus_pfad, "r", encoding="utf-8") as datei:
    for zeile in datei:
        korpus_eintraege.append(json.loads(zeile))
korpus_df = pd.DataFrame(korpus_eintraege)

# 1. Gold-Daten laden (Standard-Trennzeichen: Komma!)
gold_df = pd.read_csv(gold_pfad, encoding="utf-8-sig", dtype=str)
gold_df.columns = gold_df.columns.str.strip()

# 2. Spaltenkopf von 'id' in 'refnr' umbenennen
if "id" in gold_df.columns:
    gold_df = gold_df.rename(columns={"id": "refnr"})
else:
    gold_df = gold_df.rename(columns={gold_df.columns[0]: "refnr"})

# 3. Leerzeichen in den Werten selbst wegschneiden
gold_df["refnr"]   = gold_df["refnr"].str.strip()
korpus_df["refnr"] = korpus_df["refnr"].str.strip()

# 4. Joinen
arbeits_df = gold_df.merge(
    korpus_df[["refnr", "text"]],
    on="refnr", how="left",
)

fehlende_texte = arbeits_df["text"].isna().sum()
print(f"Gold-Anzeigen: {len(gold_df)} | Im Korpus: {len(arbeits_df) - fehlende_texte} | Ohne Text: {fehlende_texte}")
assert fehlende_texte == 0, "refnr-Mismatch — vor dem Weitermachen klären"

Gold-Anzeigen: 12 | Im Korpus: 12 | Ohne Text: 0


In [6]:
# Modell + Tokenizer laden

modell_name = RUN_INFO["modell"]

print(f"Lade Tokenizer für {modell_name} …")
tokenizer = AutoTokenizer.from_pretrained(modell_name)

Lade Tokenizer für Qwen/Qwen2.5-3B-Instruct …


/opt/conda/envs/torch/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
## Setzen des Systemprompts

system_prompt = """Du bist ein Extraktor für Stellenanzeigen.
Aus dem Anzeigentext extrahierst du strukturierte Daten und gibst sie
als gültiges JSON-Objekt zurück. Keine Prosa davor oder danach, kein
Markdown-Codeblock, ausschließlich das JSON.

Anzeigentexte können auf Deutsch oder Englisch verfasst sein. Die
Antwortwerte sind in jedem Fall die deutschen Schema-Werte unten —
niemals englische Übersetzungen davon.

Schema — extrahiere genau diese 6 Felder:

- "homeoffice": "ja", "teilweise", "nein", "remote", "nicht_genannt"
- "vertragsart": "ausbildung", "festanstellung", "praktikum", "werkstudent", "sonstiges"
- "erfahrungslevel": "junior", "mid", "senior", "egal", "nicht_genannt"
- "gehalt_min_eur": ganze Zahl (nur Euro!) oder null. Bei Fremdwährung null.
- "gehalt_zeitraum": "monat", "jahr" oder null (muss null sein, wenn gehalt_min_eur null ist)
- "skills_top3": Liste von max. 3 technischen Skills. Keine Soft Skills,
  keine Sprachen, keine Schulfächer.

Englisch → Deutsch Mapping (häufige Begriffe):

homeoffice:
  "fully remote" / "work from anywhere" → "remote"
  "hybrid" / "remote-friendly" → "teilweise"
  "on-site" / "in-office" → "nein"

vertragsart:
  "permanent" / "full-time employment" → "festanstellung"
  "internship" → "praktikum"
  "working student" / "student assistant" → "werkstudent"
  "freelance" / "trainee program" → "sonstiges"

erfahrungslevel:
  "entry-level" / "graduate" → "junior"
  "mid-level" / "intermediate" → "mid"
  "senior" / "5+ years experience" → "senior"
  "all levels welcome" → "egal"

Wähle "nicht_genannt" nur, wenn die Anzeige zum Feld tatsächlich nichts sagt.

Beispiel-Anzeige (Deutsch):
"Senior Data Engineer (m/w/d) in Vollzeit. Hybrid-Modell mit 2 Tagen Homeoffice
pro Woche. 5+ Jahre Berufserfahrung in Python und SQL erforderlich. Erfahrung
mit Snowflake oder Databricks von Vorteil. Wir bieten 70.000-85.000 € p.a."

Erwartete Ausgabe:
{"homeoffice": "teilweise", "vertragsart": "festanstellung", "erfahrungslevel": "senior", "gehalt_min_eur": 70000, "gehalt_zeitraum": "jahr", "skills_top3": ["python", "sql", "snowflake"]}

Beispiel-Anzeige (Englisch):
"Junior Data Analyst (m/f/d) — fully remote, based anywhere in Germany.
We're looking for someone with 1-2 years of experience in SQL and Python.
Knowledge of Tableau is a plus. Salary range: €45,000-55,000 per year."

Erwartete Ausgabe:
{"homeoffice": "remote", "vertragsart": "festanstellung", "erfahrungslevel": "junior", "gehalt_min_eur": 45000, "gehalt_zeitraum": "jahr", "skills_top3": ["sql", "python", "tableau"]}
"""

print(f"System-Prompt: {len(system_prompt)} Zeichen")

System-Prompt: 2532 Zeichen


In [8]:
## Hilfsfunktionen

def truncate_text(text: str, max_zeichen: int = 2000) -> str:
    """Head-Truncation auf die ersten N Zeichen."""
    if not isinstance(text, str):
        return ""
    return text[:max_zeichen]


def extract_json_robust(modell_output: str):
    """Pulls das erste {...}-Objekt aus dem Output, robust gegen Prosa-Umrandung."""
    if not modell_output:
        return None
    treffer = re.search(r"\{.*\}", modell_output, re.DOTALL)
    if not treffer:
        return None
    try:
        return json.loads(treffer.group(0))
    except json.JSONDecodeError:
        return None


def generate_extraction(anzeige_text: str) -> dict:
    """Komplette Inferenz für eine Anzeige: Prompt → generate → JSON-Parse."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": truncate_text(anzeige_text)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(modell.device)

    with torch.no_grad():
        output_ids = modell.generate(
            inputs,
            max_new_tokens=512,
            do_sample=False,                       # deterministisch
            pad_token_id=tokenizer.eos_token_id,
        )

    # Nur die neu generierten Tokens decoden (Prompt rausschneiden)
    generated  = output_ids[0][inputs.shape[1]:]
    raw_output = tokenizer.decode(generated, skip_special_tokens=True)

    return {
        "raw_output": raw_output,
        "parsed":     extract_json_robust(raw_output),
    }

In [14]:
## Inferenz-Schleife über die 12-Anzeigen

# VOR der Schleife: Einmal alle Reste aufräumen
gc.collect()
torch.cuda.empty_cache()

start_zeit = time.time()

with open(predictions_pfad, "w", encoding="utf-8") as datei:
    # 'enumerate' hinzugefügt für den Zähler 'i'
    for i, (_, zeile) in enumerate(tqdm(arbeits_df.iterrows(), total=len(arbeits_df), desc="Inferenz")):
        ergebnis = generate_extraction(zeile["text"])

        eintrag = {
            "refnr":      zeile["refnr"],
            "raw_output": ergebnis["raw_output"],   # für Debugging
            "extracted":  ergebnis["parsed"],       # None bei Parse-Fail
        }
        datei.write(json.dumps(eintrag, ensure_ascii=False) + "\n")
        
        # IN DER SCHLEIFE: Alle 50 Iterationen kurz aufräumen
        if (i + 1) % 50 == 0:
            gc.collect()
            torch.cuda.empty_cache()

dauer_sek = time.time() - start_zeit
print(f"\nFertig: {len(arbeits_df)} Anzeigen in {dauer_sek:.0f}s "
      f"({dauer_sek / len(arbeits_df):.1f}s pro Anzeige)")
print(f"Gespeichert: {predictions_pfad}")

Inferenz:   0%|          | 0/12 [00:00<?, ?it/s]

/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



Fertig: 12 Anzeigen in 60s (5.0s pro Anzeige)
Gespeichert: /home/jovyan/work/notebooks/LLM-Workshop/llm-workshop/daten/predictions_baseline.jsonl


In [15]:
## Sanity-Check der Predictions

predictions = []
with open(predictions_pfad, "r", encoding="utf-8") as datei:
    for zeile in datei:
        predictions.append(json.loads(zeile))

parse_ok    = sum(1 for p in predictions if p["extracted"] is not None)
parse_fails = [p for p in predictions if p["extracted"] is None]

print(f"JSON sauber geparst: {parse_ok}/{len(predictions)}")

# Feld-Coverage in den geparsten Predictions
felder_count = {}
for p in predictions:
    if p["extracted"]:
        for feld in p["extracted"]:
            felder_count[feld] = felder_count.get(feld, 0) + 1

erwartete_felder = ["homeoffice", "vertragsart", "erfahrungslevel",
                    "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]
print("\nFeld-Coverage:")
for feld in erwartete_felder:
    anzahl = felder_count.get(feld, 0)
    marker = "✓" if anzahl == parse_ok else "⚠"
    print(f"  {marker} {feld:22} {anzahl}/{parse_ok}")

# Unerwartete Felder (das Modell hat sich Felder ausgedacht)
unerwartet = [feld for feld in felder_count if feld not in erwartete_felder]
if unerwartet:
    print(f"\n⚠ Unerwartete Felder: {unerwartet}")

# Bei Parse-Fails: ersten Raw-Output zeigen
if parse_fails:
    print(f"\n⚠ {len(parse_fails)} Parse-Fails. Beispiel raw_output:")
    print(parse_fails[0]["raw_output"][:500])

JSON sauber geparst: 12/12

Feld-Coverage:
  ✓ homeoffice             12/12
  ✓ vertragsart            12/12
  ✓ erfahrungslevel        12/12
  ✓ gehalt_min_eur         12/12
  ✓ gehalt_zeitraum        12/12
  ✓ skills_top3            12/12


## Phase 4 — Iteration A

**Hebel: Prompt-Engineering zur Behebung der Fehler bei skills_top3 (Formatierung/Limit) und gehalt_min_eur (Bereinigung von Spannen).**

In [10]:
# PARAMETER FÜR ITERATION A
RUN_INFO_A = {
    "phase": "4.2 — Iteration A (Prompt-Klarstellung)",
    "predictions_datei": "daten/predictions_iter_A.jsonl",
}
predictions_pfad_A = basispfad / RUN_INFO_A["predictions_datei"]

system_prompt_iter_A = """Du bist ein Extraktor für Stellenanzeigen. Extrahiere ein gültiges JSON-Objekt.
Schema:
- "homeoffice": "ja", "teilweise", "nein", "remote", "nicht_genannt"
- "vertragsart": "ausbildung", "festanstellung", "praktikum", "werkstudent", "sonstiges"
- "erfahrungslevel": "junior", "mid", "senior", "egal", "nicht_genannt"
- "gehalt_min_eur": ganze Zahl oder leer lassen (null).
- "gehalt_zeitraum": "monat", "jahr" oder leer lassen (null).
- "skills_top3": Liste von max. 3 technischen Skills.

STRIKTE ANWEISUNGEN FÜR QUALITÄT:
1. SKILLS FORMATIERUNG: Schreibe alle Skills komplett klein (z.B. "python", "excel"). Keine Firmenpräfixe (schreibe "excel" statt "ms excel"). Max. 3 Skills!
2. GEHALT: Bei Spannen (z.B. 45.000 - 55.000) extrahiere NUR die Untergrenze als reine Zahl (45000). Wenn kein Gehalt dasteht, MUSS der Wert leergelassen (null) werden.
"""

def generate_A(text):
    messages = [{"role": "system", "content": system_prompt_iter_A}, {"role": "user", "content": truncate_text(text)}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(modell.device)
    with torch.no_grad():
        output_ids = modell.generate(inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=True)
    return extract_json_robust(raw)

# Run-Schleife
with open(predictions_pfad_A, "w", encoding="utf-8") as f:
    for _, zeile in tqdm(arbeits_df.iterrows(), total=len(arbeits_df), desc="Iter A"):
        f.write(json.dumps({"refnr": zeile["refnr"], "extracted": generate_A(zeile["text"])}, ensure_ascii=False) + "\n")

Iter A:   0%|          | 0/12 [00:00<?, ?it/s]

/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


## Phase 4 — Iteration B

**Hebel: Few-Shot-Beispiele einbauen (2-3 feste Beispiele im Prompt mitgeben, um das Modellverhalten radikal zu stabilisieren) + Erhöhung des Kontextfensters (max_zeichen = 4000).**

In [ ]:
# PARAMETER FÜR ITERATION B
RUN_INFO_B = {
    "phase": "4.3 — Iteration B (Few-Shot Prompting)",
    "predictions_datei": "daten/predictions_iter_B.jsonl",
}
predictions_pfad_B = basispfad / RUN_INFO_B["predictions_datei"]

# Erhöhung der Truncation auf 3000 Zeichen, um mehr Kontext zu nutzen
def truncate_text_3000(text: str) -> str:
    return text[:3000] if isinstance(text, str) else ""

system_prompt_iter_B = system_prompt_iter_A # Wir behalten die Regeln, fügen aber im Chat-Verlauf Beispiele hinzu

def generate_B(text):
    messages = [
        {"role": "system", "content": system_prompt_iter_B},
        # Few-Shot-Beispiel 1 (Deutsch)
        {"role": "user", "content": "Mitarbeiter (m/w/d) für Software-Support gesucht. Vollzeit Festanstellung, Homeoffice teilweise möglich. Erfahrung mit SQL von Vorteil."},
        {"role": "assistant", "content": '{"homeoffice": "teilweise", "vertragsart": "festanstellung", "erfahrungslevel": "nicht_genannt", "gehalt_min_eur": null, "gehalt_zeitraum": null, "skills_top3": ["sql"]}'},
        # Der echte Text
        {"role": "user", "content": truncate_text_4000(text)}
    ]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(modell.device)
    with torch.no_grad():
        output_ids = modell.generate(inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=True)
    return extract_json_robust(raw)

# Run-Schleife
with open(predictions_pfad_B, "w", encoding="utf-8") as f:
    for _, zeile in tqdm(arbeits_df.iterrows(), total=len(arbeits_df), desc="Iter B"):
        f.write(json.dumps({"refnr": zeile["refnr"], "extracted": generate_B(zeile["text"])}, ensure_ascii=False) + "\n")

Iter B:   0%|          | 0/12 [00:00<?, ?it/s]

## Phase 6 — Voller 7B-Run (gauss)

## Phase 6 — 3B-Run (euler)

In [ ]:
RUN_INFO_6_3B = {
    "predictions_datei":  "daten/predictions_full_3b.jsonl",
}
pred_pfad_3b = basispfad / RUN_INFO_6_3B["predictions_datei"]

# 1. ALTE MODELLE AUS VRAM LÖSCHEN (Zwingend notwendig, um die ~30GB VRAM freizugeben!)
if 'modell' in locals() or 'modell' in globals():
    print("Entferne Modell aus dem GPU-Speicher...")
    del modell
if 'tokenizer' in locals() or 'tokenizer' in globals():
    del tokenizer

# Grafikspeicher radikal leeren
gc.collect()
torch.cuda.empty_cache()
time.sleep(2) # Dem System kurz Zeit geben

# 2. 3B-MODELL DIREKT HIER NEU LADEN
print(f"Lade {RUN_INFO_6_3B['modell']} in FLOAT16 auf {torch.cuda.device_count()} GPU(s)...")
tokenizer = AutoTokenizer.from_pretrained(RUN_INFO_6_3B["modell"])
modell = AutoModelForCausalLM.from_pretrained(
    RUN_INFO_6_3B["modell"],
    torch_dtype=torch.float16,
    device_map="auto"
)
print("3B-Modell erfolgreich geladen und bereit für den Kontrast-Run.")


# 3. INFERENZ-SCHLEIFE ÜBER DEN VOLLEN KORPUS
# (Nutzt weiterhin deine optimierte generate_B Logik mit den Few-Shots)
with open(pred_pfad_3b, "w", encoding="utf-8") as f:
    for i, (_, zeile) in enumerate(tqdm(korpus_df.iterrows(), total=len(korpus_df), desc="Vollständiger 3B-Run")):
        ergebnis = generate_B(zeile["text"])
        
        eintrag = {
            "refnr":      zeile["refnr"],
            "extracted":  ergebnis,
        }
        f.write(json.dumps(eintrag, ensure_ascii=False) + "\n")
        
        # Regelmäßiges Aufräumen während des Runs
        if (i + 1) % 50 == 0:
            gc.collect()
            torch.cuda.empty_cache()

print(f"Fertig! 3B-Predictions erfolgreich gespeichert unter: {pred_pfad_3b}")